# Data Preperation

In [ ]:
import glob
import os

#ud-treebanks-v2.18 folder shoud be placeed directly under the Lab2 folder

#Windows vertion
#UD_DIR = r"ud-treebanks-v2.18\ud-treebanks-v2.18"

#Mac vertion
UD_DIR = r"../ud-treebanks-v2.18/ud-treebanks-v2.18"

os.makedirs("../data", exist_ok=True)

# 1. Read the languages from list.txt
with open("../list.txt", encoding="utf-8") as f:
    languages = [line.strip().split(" ")[0] for line in f if line.strip() if line.strip() and not line.startswith("language")]
print(f"Languages: {', '.join(languages)}")

from conllu import parse_incr
import networkx as nx

def build_network(paths, use="form", skip_punct=False, lower=True): #ask if form or lemma, if lower or not
    G = nx.DiGraph()
    for path in paths:
        with open(path, encoding="utf-8") as f:
            for sentences in parse_incr(f):
                # keep only real tokens: skip multiword (individual words are part of a multi-word token) and empty nodes (8.1)
                types = {t["id"]: t for t in sentences if isinstance(t["id"], int)}

                for type in types.values():
                    head_id = type["head"]
                    if head_id == 0:# root has no head token - no edge to add
                        continue
                    if skip_punct and type["upos"] == "PUNCT": # possibly test
                        continue
                    head = types.get(head_id)
                    if head is None:
                        print(f"Warning: head {head_id} not found for token {type['id']} in file {path}")
                        continue

                    h, d = head[use], type[use]
                    if h is None or d is None or h == "_" or d == "_":
                        # skip tokens with missing edges
                        #print(f"Warning: missing edge for token {type['id']} in file {path} (head: {h}, dep: {d}), type: {type}")
                        continue
                    if lower: # possible test
                        h, d = h.lower(), d.lower()
                    #print(f"Adding edge: {h} -> {d}")
                    if G.has_edge(h, d):
                        G[h][d]["weight"] += 1
                    else:
                        G.add_edge(h, d, weight=1)
    return G


for language in languages:

    # Select all the treebanks 
    files = sorted(glob.glob(os.path.join(UD_DIR, f"UD_{language}-*", "*.conllu"))) # finds dev, test and train files for each language
    print(f"Language: {language}, Treebanks: {len(files)}, Files: {files}")
    # Build the network
    G = build_network(files)

    # extract degree sequence ! out-degree, also remove nodes with degree 0
    degree_sequence = sorted([d for n, d in G.out_degree() if d > 0])

    #save degree sequence to file
    with open(f"../data/{language}_degree_sequence.txt", "w") as f:
        f.write("\n".join(map(str, degree_sequence)))
    print(f"Degree sequence for {language} saved to ../data/{language}_degree_sequence.txt")
